### Authorship
@author: Alexandre Pereira Santos <br>
alexandre.santos(at)lmu.de<br>
- uses OSMNx, pandas, geopandas, rasterio
- may include content from ChatGPT or GitHub Copilot

### Features
- Prepares the SEP assignment at the pixel level, starting from the admin level2 units
- Results should be similar to the ABM outputs. This means having a raster with integers classifing each pixel in:
  - 0 = non-urban (excluded from analysis)
  - 1 = deprived SEP
  - 2 = low SEP
  - 3 = middle SEP
  - 4 = high SEP
- Each HDI component at the admin 2 level (regencies and cities) will be downscaled using a spatial covariate:
  - Education (EYE2015 and EYE2025): case_city + '_LOC_schools_OSM_2025_distance_normal_150m.tif'
  - Life expectancy (LEB2015 and LEB2025): case_city + '_LOC_health_OSM_2025_distance_normal_150m.tif'
  - Adjusted expenditure 2015: external_path / f"{case_city}_ECO_GDP_PPP_1990_2024_Kummu_etal.tif" (band 6)
  - Adjusted expenditure 2025: external_path / f"{case_city}_ECO_GDP_PPP_1990_2024_Kummu_etal.tif" (band 8)
- Addtionally, we provide two binary classifications of urban:
  - Urban binary 2015: case_city + f'_URB_urbanisation_2015_GHSL_150m.tif'
  - Urban binary 2025: case_city + f'_URB_urbanisation_2025_GHSL_150m.tif'
  


### Prerequisites
- AOI vector file (i.e., shapefile or geopackage)
- Raster reference file with the same extent as the AOI

## Roadmap:


# Assumptions:
| Feature | High SEP (4) | Mid SEP (3) | Low SEP (2) | Deprived SEP (1) |
| --- | --- | --- | --- | --- |
| Distance to health | ⬇️ Low | ⬇️ Low | ⬆️ High | ⬆️ High |
| Distance to schools | ⬇️ Low | ⬇️ Low | ⬆️ High | ⬆️ High |
| GDP | ⬆️ High | ⬆️ Medium | ⬇️ Low | ⬇️ Lowest |
| Urban surface | ⬆️ High | ⬆️ Medium | ⬇️ Low | ⬇️ Lowest |

**These assumptions should lead to weight attributions**
| Feature Type | Weight | Rationale |
| --- | --- | --- |
| HDI components (LEB, EYE, AEP) | 0.1–0.3 | Low weight: Provides admin2 baseline but allows covariates to refine it. |
| Distance covariates (health, schools) | -1.0–1.5 | Negative: Shorter distances = higher SEP. Higher magnitude = stronger influence. |
| GDP/Urban surface | 1.0–1.5 | Positive: Higher values = higher SEP. |

When translating into weights, please consider:
**Risks of Excessive Weights (>2.0)**<br>
- Feature domination: A single feature (e.g., GDP with weight=3.0) can override all others, reducing clustering to a 1D problem.<br>
Example: If GDP weight=3.0 and HDI weights=0.2, GDP will dominate 95% of the distance calculation.
- Loss of HDI downscaling: If covariate weights are too high, the HDI baseline is ignored, defeating the purpose of downscaling.
- Numerical instability: Very high weights (e.g., >5.0) can cause floating-point precision issues in distance calculations.

About **coord_weight**

| Value | Spatial Influence | Use Case |
| --- | --- | --- |
| 0.0 | None | Pure feature-based clustering (ignores geography) |
| 0.1–0.2 | Mild | Default – Balances features and spatial coherence |
| 0.3–0.5 | Strong | Enforces contiguous clusters (e.g., for urban planning) |
| >0.5 | Dominant | Spatial coordinates override features (rarely useful) |

Colour settings for the SEPs<br>
- 'Deprived'[1]: #f0a1c5
- 'Low'[2]: #6eca74
- 'Mid'[3]: #98abf7
- 'High'[4]: #414bb2 


# imports

In [1]:
# utils
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# math
import numpy as np

# geospatial
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.mask import mask

# machine learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.ndimage import generic_filter

# built-in functions
from ti_city_00_raster_functions import *

# init

In [3]:
#  the 'case_city' variable defines the city we are working on. 
# APS: Need to update the code below to change it from MUM to the variable name

case_city = 'JAK' #MUM MAN JAK
drop_path = r'C:\\Sciebo\\05_GIS\\' # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\GIS' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\GIS'
external_data_path = Path(r'C:\\Sciebo\\00_data') # adjust this to match the path on your computer, e.g. 'D:\\Dropbox\\x\\PostDoc\\00_data' or 'C:\\Users\\x\\Dropbox\\x\\PostDoc\\00_data'
## ascii150_arcgis_path = Path(r"D:\\Dropbox\\x\\PostDoc\ASCII_150m") deprecated, ignore

#input a vector and a raster file for each city
AOI_path = Path(f'{drop_path}\\{case_city}\\processed\\')
AOI_file = f'{case_city}_LIM_AOI_reference_150m_A.shp' # APS: updated AOI 01.08.2024

AOI_gdf = gpd.read_file(AOI_path / AOI_file) #.to_crs(epsg=4326)
AOI_gdf_4326 = AOI_gdf.to_crs(epsg=4326) 

ref_raster_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')
ref_raster_150_path = Path(f'../data/processed/{case_city}_LIM_reference_AOI_150m.tif')

raw_path = Path(f'\\{case_city}\\raw\\')
interim_path = Path(drop_path + f'{case_city}\\interim\\')
processed_path = Path(drop_path + f'{case_city}\\processed\\')
external_path = Path(drop_path + f'{case_city}\\external\\')
model_inputs_30m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_30m\\')
model_inputs_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_150m\\')
model_inputs_SLEUTH_150m_path = Path(drop_path + f'{case_city}\\model_inputs\\TIFF_SLEUTH_150m')
ascii150_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_150m')
ascii150_SLEUTH_path = Path(drop_path + f'\\{case_city}\\model_inputs\\ASCII_SLEUTH_150m')
ti_city_ascii_path = Path(f'../model/{case_city}/data/in/')
ti_city_out_path = Path(f'../model/{case_city}/data/out/')
ref_raster_sleuth_path = model_inputs_SLEUTH_150m_path / (case_city + '_URB_SLEUTH_input_2022.tif' )

if case_city == 'MUM':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]

if case_city == 'MAN':    
    gadm_var = 'NAME_2'
    calibration_years = [2015,2025]

if case_city == 'JAK':
    gadm_var = 'NAME_3'
    calibration_years = [2015,2025]

#%run ./ti_city_00_raster_functions.ipynb

# read the reference raster
with rasterio.open(ref_raster_path,'r') as src: # APS 20.08.2025 using the 150 m raster
    ref_raster = src
    ref_meta = src.meta
    ref_height, ref_width, ref_area = get_transform(ref_raster)

# suppress deprecation warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

# print the AOI bound coordinates to test impo
bounds = AOI_gdf.total_bounds
# print the AOI bound coordinates
bounds = AOI_gdf.total_bounds
print("AOI bounds:")
print(f"| {'lon_min':<10} | {'lat_min':<10} |")
print(f"| {'lon_max':<10} | {'lat_max':<10} |")
print(f"| {bounds[0]:<10.4f} | {bounds[1]:<10.4f} |")
print(f"| {bounds[2]:<10.4f} | {bounds[3]:<10.4f} |")


AOI bounds:
| lon_min    | lat_min    |
| lon_max    | lat_max    |
| 639000.0000 | 9249000.0000 |
| 754800.0000 | 9351000.0000 |


## Helper functions

In [4]:
def add_raster_column(grid_gdf, raster_path, col_name):
    """
    Extract values from a **single-band raster** at grid cell centroids.
    Adds a column `col_name` to `grid_gdf` with the sampled values.
    """
    with rasterio.open(raster_path) as src:
        coords = [(geom.centroid.x, geom.centroid.y) for geom in grid_gdf.geometry]
        samples = list(src.sample(coords))
        values = [s[0] if s else np.nan for s in samples]  # Extract band 1
    grid_gdf[col_name] = values
    return grid_gdf

def add_raster_band_column(grid_gdf, raster_path, band, col_name):
    """
    Extract values from a **specific band** of a multi-band raster.
    """
    with rasterio.open(raster_path) as src:
        coords = [(geom.centroid.x, geom.centroid.y) for geom in grid_gdf.geometry]
        samples = list(src.sample(coords, indexes=[band]))
        values = [s[0] if s else np.nan for s in samples]
    grid_gdf[col_name] = values
    return grid_gdf

''' un-weighted version of prepare_clustering_data
def prepare_clustering_data(grid_gdf, year, gdp_band):
    """
    Prepare data for clustering for a given year.
    Returns:
        urban_gdf: GeoDataFrame of urban cells only
        features_scaled: Standardized feature matrix (n_urban_cells × n_features)
        urban_mask: Boolean mask for urban cells in original grid_gdf
        feature_cols: List of feature column names
    """
    # Filter urban cells (1 = urban, 2 = non-urban)
    urban_mask = grid_gdf[f'urban_binary_{year}'] == 1
    urban_gdf = grid_gdf[urban_mask].copy()

    # Define feature columns (HDI components + covariates)
    feature_cols = [
        f'LEB{year}', f'EYE{year}', f'AEP{year}',
        'health_dist', 'school_dist', f'gdp_{year}',
        f'urban_surface_{year}'
    ]

    # Fill missing values with column means (critical for robustness)
    urban_gdf[feature_cols] = urban_gdf[feature_cols].fillna(
        urban_gdf[feature_cols].mean()
    )

    # Standardize features (K-means is sensitive to scale)
    features = urban_gdf[feature_cols].values
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    return urban_gdf, features_scaled, urban_mask, feature_cols'''

def prepare_clustering_data(grid_gdf, year, feature_weights=None):
    urban_mask = grid_gdf[f'urban_binary_{year}'] == 1
    urban_gdf = grid_gdf[urban_mask].copy()

    # Define features and weights
    feature_cols = [
        f'LEB{year}', f'EYE{year}', f'AEP{year}',  # HDI components (admin2)
        'health_dist', 'school_dist',             # Distance covariates
        f'gdp_{year}', f'urban_surface_{year}'   # Economic/urban covariates
    ]

    # Feature weights: HDI << covariates
    if feature_weights is None: # if no weights were passed, assume equal
        feature_weights = {
            f'LEB{year}': 1.0, f'EYE{year}': 1.0, f'AEP{year}': 1.0,  
            'health_dist': -1.0, 'school_dist': -1.0,     # negative: shorter = better            
            f'gdp_{year}': 1.0, f'urban_surface_{year}': 1.0         
        }

    # Fill missing values and standardize
    urban_gdf[feature_cols] = urban_gdf[feature_cols].fillna(urban_gdf[feature_cols].mean())
    features = urban_gdf[feature_cols].values
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Apply weights
    for i, col in enumerate(feature_cols):
        features_scaled[:, i] *= feature_weights[col]

    return urban_gdf, features_scaled, urban_mask, feature_cols

def spatial_kmeans(features, coords, n_clusters=4, random_state=42, coord_weight=0.2):
    """
    Spatial-aware K-means: Augments feature space with scaled coordinates.
    Args:
        coord_weight: Weight for spatial coordinates (tune between 0.1–0.5).
                     Higher = more spatially contiguous clusters.
    Returns:
        labels: Cluster assignments (0 to n_clusters-1)
        kmeans: Fitted KMeans model
    """
    if len(features) < n_clusters:
        warnings.warn(f"Fewer urban cells ({len(features)}) than clusters ({n_clusters}). Using all as separate clusters.")
        return np.arange(len(features)), None

    # Scale coordinates to match feature ranges
    coord_scaler = StandardScaler()
    coords_scaled = coord_scaler.fit_transform(coords)

    # Combine features + weighted coordinates
    augmented_features = np.hstack([features, coords_scaled * coord_weight])

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = kmeans.fit_predict(augmented_features)
    return labels, kmeans

def sort_clusters_by_hdi(urban_gdf, labels, year):
    """Reorder clusters so Class 4 = highest HDI."""
    #print(labels)
    cluster_hdi = []
    unique_labels = np.unique(labels)
    for label in unique_labels:
        mask = labels == label
        mean_hdi = urban_gdf.loc[mask, f'HDI{year}'].mean()
        cluster_hdi.append(mean_hdi)
    
    # Sort clusters: [deprived, low, mid, high] → [1, 2, 3, 4]
    sorted_indices = np.argsort(cluster_hdi)  # Ascending HDI order
    label_mapping = {old: new for new, old in enumerate(sorted_indices)}
    #label_mapping = {old: new+1 for new, old in enumerate(sorted_indices)}
    return np.array([label_mapping[label] for label in labels])

def rasterize_grid(grid_gdf, sep_col, ref_raster_path, output_path, export=True):
    """
    Rasterize grid_gdf with SEP classes to match reference raster metadata.
    Non-urban cells (SEP=0) are preserved.
    """
    with rasterio.open(ref_raster_path) as src:
        meta = src.meta.copy()
        transform = src.transform
        height, width = src.height, src.width

    # Create (geometry, value) pairs
    shapes = [(geom, value) for geom, value in zip(grid_gdf.geometry, grid_gdf[sep_col])]

    # Rasterize
    raster = rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,  # Default value for non-urban
        dtype='uint8'
    )

    # Update metadata
    meta.update(dtype=raster.dtype, count=1, nodata=0)
    
    if export:
        # Write output
        with rasterio.open(output_path, 'w', **meta) as dst:
            dst.write(raster, 1)
            return output_path

def majority_filter(raster, size=3):
    """Apply majority filter to reduce salt-and-pepper noise."""
    return generic_filter(
        raster, lambda x: np.bincount(x).argmax(), size=size, mode='constant', cval=0
    )

'''def print_cluster_centers(kmeans, feature_cols, year):
    """Print cluster centers to interpret SEP classes."""
    centers = pd.DataFrame(
        kmeans.cluster_centers_,
        columns=feature_cols
    )
    # Sort by HDI (average of components) to map to SEP classes
    hdi_cols = [col for col in feature_cols if col.startswith(('LEB', 'EYE', 'AEP'))]
    centers['HDI_Mean'] = centers[hdi_cols].mean(axis=1)
    centers = centers.sort_values('HDI_Mean', ascending=False)
    
    print(f"\n📊 {year} Cluster Centers (Sorted by HDI):")
    print(centers.to_string())
    print("\nSuggested SEP class mapping (highest HDI → lowest):")
    for i, (_, row) in enumerate(centers.iterrows()):
        print(f"  Cluster {i} → SEP Class {4-i}")  # 4=High, 1=Deprived'''

def print_cluster_centers(kmeans, feature_cols, year):
    """Print cluster centers to interpret SEP classes."""
    if kmeans is None:
        print(f"\n📊 {year} KMeans model not available.")
        return

    centers = kmeans.cluster_centers_
    n_features = len(feature_cols)

    if centers.shape[1] > n_features:
        centers = centers[:, :n_features]

    centers_df = pd.DataFrame(centers, columns=feature_cols)

    hdi_cols = [col for col in feature_cols if col.startswith(('LEB', 'EYE', 'AEP'))]
    centers_df['HDI_Mean'] = centers_df[hdi_cols].mean(axis=1)
    centers_df = centers_df.sort_values('HDI_Mean', ascending=False)

    print(f"\n📊 {year} Cluster Centers (Sorted by HDI):")
    print(centers_df.to_string())
    print("\nSuggested SEP class mapping (highest HDI → lowest):")
    for i, (_, row) in enumerate(centers_df.iterrows()):
        print(f"  Cluster {i} → SEP Class {4-i}")

## Main functions

### Load data

In [5]:
# =============================================
# 1️⃣ LOAD DATA (Grid + Rasters)
# =============================================
def load_data(case_city, interim_path, external_path, calibration_years):
    """
    Load grid data and extract raster values for both years.
    Returns:
        grid_2015: GeoDataFrame with raster values for early year
        grid_2025: GeoDataFrame with raster values for late year
    """
    print("  🔍 Loading grid data...")
    grid_gdf = gpd.read_file(
        interim_path / f"{case_city}_LIM_grid_150m_identify_admin_level2_A.shp",
        usecols=[
            'ADM2_BA', 'ADM2_EN', 'ADM2CODE', 'HDI2025', 'HDI2020', 'HDI2015',
            'LEB2025', 'EYE2025', 'AEP2025', 'LEB2020', 'EYE2020', 'AEP2020',
            'LEB2015', 'EYE2015', 'AEP2015',  'grid_area', #'Shape_Leng',
            'LVL2_AREA', 'LVL2_PROP', 'geometry'
        ]
    )
    #grid_gdf.drop(columns=['Shape_Leng'], inplace=True)
    #grid_gdf['LVL2_PROP'] = grid_gdf['grid_area'] / grid_gdf['LVL2_AREA'].astype(float)

    # Define raster paths
    builtup_raster_early_url = interim_path / f"{case_city}_URB_urbanisation_{calibration_years[0]}_GHSL_150m.tif"
    builtup_raster_late_url = interim_path / f"{case_city}_URB_urbanisation_{calibration_years[1]}_GHSL_150m.tif"
    builtup_surface_raster_early_url = interim_path / f"{case_city}_URB_urban_surface_{calibration_years[0]}_GHSL_150m.tif"
    builtup_surface_raster_late_url = interim_path / f"{case_city}_URB_urban_surface_{calibration_years[1]}_GHSL_150m.tif"
    life_expectancy_covar_url = interim_path / f"{case_city}_LOC_health_OSM_2025_distance_normal_150m.tif"
    education_covar_url = interim_path / f"{case_city}_LOC_schools_OSM_2025_distance_normal_150m.tif"
    gdp_path = external_path / f"{case_city}_ECO_GDP_PPP_1990_2024_Kummu_etal.tif"
    gdp_bands = [6, 8]  # Band 6 = 2015, Band 8 = 2024/2025

    # Add raster values for 2015
    print("  📥 Extracting raster values for 2015...")
    grid_2015 = (
        grid_gdf.copy()
        .pipe(add_raster_column, builtup_raster_early_url, 'urban_binary_2015')
        .pipe(add_raster_column, builtup_surface_raster_early_url, 'urban_surface_2015')
        .pipe(add_raster_column, life_expectancy_covar_url, 'health_dist')
        .pipe(add_raster_column, education_covar_url, 'school_dist')
        .pipe(add_raster_band_column, gdp_path, gdp_bands[0], 'gdp_2015')
    )

    # Add raster values for 2025
    print("  📥 Extracting raster values for 2025...")
    grid_2025 = (
        grid_gdf.copy()
        .pipe(add_raster_column, builtup_raster_late_url, 'urban_binary_2025')
        .pipe(add_raster_column, builtup_surface_raster_late_url, 'urban_surface_2025')
        .pipe(add_raster_column, life_expectancy_covar_url, 'health_dist')
        .pipe(add_raster_column, education_covar_url, 'school_dist')
        .pipe(add_raster_band_column, gdp_path, gdp_bands[1], 'gdp_2025')
    )

    return grid_2015, grid_2025



### Cluster and assign SEP classes

In [9]:
# =============================================
# 2️⃣ CLUSTER URBAN CELLS
# =============================================

def cluster_urban_cells(grid_2015, grid_2025, calibration_years, n_clusters=4, coord_weight=0.2, feature_weights=None):
    """
    Cluster urban cells for both years using spatial K-means.
    Returns:
        dict: {
            'labels_2015': array, 'kmeans_2015': model, 'urban_mask_2015': bool array,
            'feature_cols_2015': list, 'labels_2025': array, 'kmeans_2025': model,
            'urban_mask_2025': bool array, 'feature_cols_2025': list
        }
    """
    print("  🎯 Clustering 2015...")
    # Cluster 2015
    urban_2015, features_2015, urban_mask_2015, feature_cols_2015 = prepare_clustering_data(
        grid_2015, calibration_years[0], feature_weights
    )
    coords_2015 = np.column_stack([urban_2015.geometry.centroid.x, urban_2015.geometry.centroid.y])
    labels_2015, kmeans_2015 = spatial_kmeans(features_2015, coords_2015, n_clusters=n_clusters, coord_weight=coord_weight)
    labels_2015 = sort_clusters_by_hdi(urban_2015, labels_2015, calibration_years[0])  # 👈 Sort by HDI

    print("  🎯 Clustering 2025...")
    # Cluster 2025 (same logic)
    urban_2025, features_2025, urban_mask_2025, feature_cols_2025 = prepare_clustering_data(
        grid_2025, calibration_years[1], feature_weights
    )
    coords_2025 = np.column_stack([
        urban_2025.geometry.centroid.x, 
        urban_2025.geometry.centroid.y
        ])
    labels_2025, kmeans_2025 = spatial_kmeans(features_2025, coords_2025, n_clusters=n_clusters, coord_weight=coord_weight)
    labels_2025 = sort_clusters_by_hdi(urban_2025, labels_2025, calibration_years[1])  # 👈 Sort by HDI

    return {
        'labels_2015': labels_2015, 'kmeans_2015': kmeans_2015,
        'urban_mask_2015': urban_mask_2015, 'feature_cols_2015': feature_cols_2015,
        'labels_2025': labels_2025, 'kmeans_2025': kmeans_2025,
        'urban_mask_2025': urban_mask_2025, 'feature_cols_2025': feature_cols_2025

    }

'''def cluster_urban_cells(grid_2015, grid_2025, calibration_years, coord_weight=0.2, feature_weights=None):
    """
    Cluster urban cells for both years using spatial K-means.
    Returns:
        dict: {
            'labels_2015': array, 'kmeans_2015': model, 'urban_mask_2015': bool array,
            'feature_cols_2015': list, 'labels_2025': array, 'kmeans_2025': model,
            'urban_mask_2025': bool array, 'feature_cols_2025': list
        }
    """
    print("  🎯 Clustering 2015...")
    urban_2015, features_2015, urban_mask_2015, feature_cols_2015 = prepare_clustering_data(
        grid_2015, calibration_years[0], feature_weights
    )
    coords_2015 = np.column_stack([
        urban_2015.geometry.centroid.x,
        urban_2015.geometry.centroid.y
    ])
    labels_2015, kmeans_2015 = spatial_kmeans(
        features_2015, coords_2015, n_clusters=n_clusters, random_state=42, coord_weight=coord_weight
    )

    print("  🎯 Clustering 2025...")
    urban_2025, features_2025, urban_mask_2025, feature_cols_2025 = prepare_clustering_data(
        grid_2025, calibration_years[1], feature_weights
    )
    coords_2025 = np.column_stack([
        urban_2025.geometry.centroid.x,
        urban_2025.geometry.centroid.y
    ])
    labels_2025, kmeans_2025 = spatial_kmeans(
        features_2025, coords_2025, n_clusters=n_clusters, random_state=42, coord_weight=coord_weight
    )

    return {
        'labels_2015': labels_2015, 'kmeans_2015': kmeans_2015,
        'urban_mask_2015': urban_mask_2015, 'feature_cols_2015': feature_cols_2015,
        'labels_2025': labels_2025, 'kmeans_2025': kmeans_2025,
        'urban_mask_2025': urban_mask_2025, 'feature_cols_2025': feature_cols_2025
    }'''


# =============================================
# 3️⃣ ASSIGN SEP CLASSES
# =============================================
def assign_sep_classes(grid_2015, grid_2025, cluster_results):
    """
    Assign SEP classes (0=non-urban, 1-4=socioeconomic profiles) to grid cells.
    Returns:
        grid_2015: GeoDataFrame with 'SEP_2015' column
        grid_2025: GeoDataFrame with 'SEP_2025' column
    """
    print("  📊 Assigning SEP classes...")
    grid_2015 = grid_2015.copy()
    grid_2025 = grid_2025.copy()

    # Assign classes: 0 (non-urban), 1-4 (urban clusters)
    grid_2015['SEP_2015'] = 0
    grid_2015.loc[cluster_results['urban_mask_2015'], 'SEP_2015'] = cluster_results['labels_2015'] + 1

    grid_2025['SEP_2025'] = 0
    grid_2025.loc[cluster_results['urban_mask_2025'], 'SEP_2025'] = cluster_results['labels_2025'] + 1

    return grid_2015, grid_2025


### Export & print

In [10]:
# =============================================
# 4️⃣ PRINT & EXPORT (Controlled by Flag)
# =============================================
def export_results(
    grid_2015, grid_2025, cluster_results, case_city, interim_path,
    calibration_years, ref_raster_path, output_suffix, export_files=False
):
    """
    Print cluster centers and export rasters if export_files=True.
    Returns:
        Tuple of output paths (or (None, None) if export_files=False)
    """
    print("  📄 Printing cluster centers...")
    print_cluster_centers(cluster_results['kmeans_2015'], cluster_results['feature_cols_2015'], calibration_years[0])
    print_cluster_centers(cluster_results['kmeans_2025'], cluster_results['feature_cols_2025'], calibration_years[1])

    print("  💾 Rasterizing results...")
    output_2015 = interim_path / f"{case_city}_SEP_{calibration_years[0]}_150m{output_suffix}.tif"
    output_2025 = interim_path / f"{case_city}_SEP_{calibration_years[1]}_150m{output_suffix}.tif"

    rasterize_grid(grid_2015, 'SEP_2015', ref_raster_path, output_2015, export=export_files)
    rasterize_grid(grid_2025, 'SEP_2025', ref_raster_path, output_2025, export=export_files)

    if export_files:
        print(f"  ✅ Outputs saved to:\n     - {output_2015}\n     - {output_2025}")
    else: 
        print("  ⏭️  Skipping export (export_files=False)")
        #return None, None
        
    return output_2015, output_2025


### Main workflow (orchestrator)

In [11]:
def run_hdi_downscaling(
    case_city,
    interim_path,
    external_path,
    calibration_years,
    ref_raster_path,
    AOI_gdf,
    coord_weight=0.2,
    n_clusters=4,
    output_suffix="",
    export_files=False, 
    feature_weights=None
):
    """
    End-to-end workflow for HDI downscaling and SEP classification.

    Args:
        export_files: bool (default=False)
            - False: Skip printing/exporting (for debugging)
            - True: Print cluster centers and save rasters
    Returns:
        Tuple of output paths if export_files=True, else (None, None)
    """
    print("🔍 Loading data...")
    grid_2015, grid_2025 = load_data(case_city, interim_path, external_path, calibration_years)

    print("🎯 Clustering urban cells...")
    cluster_results = cluster_urban_cells(grid_2015, grid_2025, calibration_years, n_clusters=n_clusters, coord_weight=coord_weight, feature_weights=feature_weights)

    print("📊 Assigning SEP classes...")
    grid_2015, grid_2025 = assign_sep_classes(grid_2015, grid_2025, cluster_results)

    print("💾 Exporting results...")
    return export_results(
        grid_2015, grid_2025, cluster_results, case_city, interim_path,
        calibration_years, ref_raster_path, output_suffix, export_files
    )

# 1. HDI downscaling

In [ ]:
#Example usage:
output_2015, output_2025 = run_hdi_downscaling(
    case_city="Jakarta",
    interim_path=interim_path,
    external_path=external_path,
    calibration_years=calibration_years,
    ref_raster_path=ref_raster_path,
    AOI_gdf=AOI_gdf,
    coord_weight=0.2,  # Tune this for spatial coherence
    n_clusters=4,  # Number of SEP classes
    export_files=True,  # Set to True to save outputs
    feature_weights=None,  # Use default weights if None
    output_suffix="_v1"
    )

In [72]:
# step-by-step debugging:
# step 1: Load data

grid_2015, grid_2025 = load_data(
    case_city = case_city, 
    interim_path = interim_path, 
    external_path = external_path, 
    calibration_years = calibration_years)

  🔍 Loading grid data...
  📥 Extracting raster values for 2015...
  📥 Extracting raster values for 2025...


In [73]:
print(grid_2015[['urban_binary_2015', 'gdp_2015', 'HDI2015']].head())

   urban_binary_2015   gdp_2015  HDI2015
0                1.0  5441720.0    70.05
1                NaN  4865749.5    70.05
2                NaN  4290626.0    70.05
3                NaN  4993897.0    70.05
4                1.0  5643741.5    70.05


In [74]:
print(grid_2025[['urban_binary_2025', 'gdp_2025', 'HDI2025']].head())

   urban_binary_2025    gdp_2025  HDI2025
0                1.0  10954296.0    76.19
1                NaN   9997255.0    76.19
2                NaN   9041910.0    76.19
3                NaN  10353638.0    76.19
4                1.0  11504935.0    76.19


## introducing weights

In [93]:
# step 2: clustering
feature_weights = {
    f'LEB{calibration_years[0]}': 0.2,            # Low weight for HDI
    f'EYE{calibration_years[0]}': 0.2,            # Low weight for HDI
    f'AEP{calibration_years[0]}': 0.2,            # Low weight for HDI 
    f'gdp_{calibration_years[0]}': 1.3,           # Positive: higher = better
    f'urban_surface_{calibration_years[0]}': 1.0, # Positive: higher = better
    f'LEB{calibration_years[1]}': 0.2,            # Low weight for HDI
    f'EYE{calibration_years[1]}': 0.2,            # Low weight for HDI
    f'AEP{calibration_years[1]}': 0.2,            # Low weight for HDI
    f'gdp_{calibration_years[1]}': 1.3,           # Positive: higher = better
    f'urban_surface_{calibration_years[1]}': 1.0, # Positive: higher = better
    'health_dist': -1.2,                          # Negative: shorter = better
    'school_dist': -1.2                           # Negative: shorter = better
}
cluster_results_dict = cluster_urban_cells(grid_2015, grid_2025, calibration_years, n_clusters=4, coord_weight=0.2, feature_weights=feature_weights)
print(f"Unique labels for {calibration_years[0]}: {np.unique(cluster_results_dict['labels_2015'])} (should be [0,1,2,3])")
print(f"Unique labels for {calibration_years[1]}: {np.unique(cluster_results_dict['labels_2025'])} (should be [0,1,2,3])")

  🎯 Clustering 2015...
[0 0 0 ... 3 3 3]
  🎯 Clustering 2025...
[2 2 2 ... 3 3 3]
Unique labels for 2015: [0 1 2 3] (should be [0,1,2,3])
Unique labels for 2025: [0 1 2 3] (should be [0,1,2,3])


In [94]:
# step 3: assign SEP classes
SEP_grid_2015, SEP_grid_2025 = assign_sep_classes(grid_2015, grid_2025, cluster_results_dict)
print(f"SEP class distribution for {calibration_years[0]}: {SEP_grid_2015['SEP_2015'].value_counts()}")  # Count cells per class
print(f"SEP class distribution for {calibration_years[1]}: {SEP_grid_2025['SEP_2025'].value_counts()}")  # Count cells per class

  📊 Assigning SEP classes...
SEP class distribution for 2015: SEP_2015
3    51943
2    50146
0    43239
4    24285
1    11582
Name: count, dtype: int64
SEP class distribution for 2025: SEP_2025
2    53433
3    52928
0    36135
4    26255
1    12444
Name: count, dtype: int64


In [ ]:
# step 4: visualise and export results (optional)
export_results(
    SEP_grid_2015, 
    SEP_grid_2025, 
    cluster_results_dict, 
    case_city = case_city, 
    interim_path = interim_path,
    calibration_years = calibration_years, 
    ref_raster_path = ref_raster_path, 
    output_suffix = "_v04", 
    export_files=True
)

  📄 Printing cluster centers...

📊 2015 Cluster Centers (Sorted by HDI):
    LEB2015   EYE2015   AEP2015  health_dist  school_dist  gdp_2015  urban_surface_2015  HDI_Mean
1  0.123260  0.138939  0.261700     0.906825     0.580363  2.502143            0.888453  0.174633
2  0.055917  0.068798 -0.003134     0.482598     0.521689 -0.253204            0.602925  0.040527
3 -0.069799 -0.092700 -0.103106    -0.277598    -0.198757 -0.737212           -0.867439 -0.088535
0 -0.205959 -0.197330 -0.087596    -2.853036    -2.685840 -0.914282           -0.803184 -0.163628

Suggested SEP class mapping (highest HDI → lowest):
  Cluster 0 → SEP Class 4
  Cluster 1 → SEP Class 3
  Cluster 2 → SEP Class 2
  Cluster 3 → SEP Class 1

📊 2025 Cluster Centers (Sorted by HDI):
    LEB2025   EYE2025   AEP2025  health_dist  school_dist  gdp_2025  urban_surface_2025  HDI_Mean
1  0.127879  0.064364  0.297631     0.907816     0.606132  2.444320            0.925801  0.163291
0  0.066143  0.082918 -0.007084     0.50515

(WindowsPath('C:/Sciebo/05_GIS/JAK/interim/JAK_SEP_2015_150m_v04.tif'),
 WindowsPath('C:/Sciebo/05_GIS/JAK/interim/JAK_SEP_2025_150m_v04.tif'))

In [ ]:
# alternative: full workflow implementation
output_2015, output_2025 = run_hdi_downscaling(
    case_city=case_city,
    interim_path=interim_path,
    external_path=external_path,
    calibration_years=calibration_years,
    ref_raster_path=ref_raster_path,
    AOI_gdf=AOI_gdf,
    coord_weight=0.6,  # Tune this for spatial coherence
    output_suffix="_v03", 
    export_files=True
    )

🔍 Loading data...
  🔍 Loading grid data...
  📥 Extracting raster values for 2015...
  📥 Extracting raster values for 2025...
🎯 Clustering urban cells...
  🎯 Clustering 2015...
  🎯 Clustering 2025...
📊 Assigning SEP classes...
  📊 Assigning SEP classes...
💾 Exporting results...
  📄 Printing cluster centers...

📊 2015 Cluster Centers (Sorted by HDI):
    LEB2015   EYE2015   AEP2015  health_dist  school_dist  gdp_2015  urban_surface_2015  HDI_Mean
2  0.615962  0.686634  1.398145    -0.754165    -0.476649  1.844229            0.892311  0.900247
0  0.669466  0.840018  0.257327    -0.409636    -0.482719 -0.120079            0.319705  0.588937
3  0.032155 -0.375475 -1.017638     0.250250     0.234043 -0.613958           -0.521323 -0.453653
1 -2.100683 -1.581464  0.170672     1.115668     0.964465 -0.553219           -0.485244 -1.170492

Suggested SEP class mapping (highest HDI → lowest):
  Cluster 0 → SEP Class 4
  Cluster 1 → SEP Class 3
  Cluster 2 → SEP Class 2
  Cluster 3 → SEP Class 1

📊

## sensitivity testing

In [12]:
# alternative: full workflow implementation
feature_weights = {
    f'LEB{calibration_years[0]}': 0.2,            # Low weight for HDI
    f'EYE{calibration_years[0]}': 0.2,            # Low weight for HDI
    f'AEP{calibration_years[0]}': 0.2,            # Low weight for HDI 
    f'gdp_{calibration_years[0]}': 1.3,           # Positive: higher = better
    f'urban_surface_{calibration_years[0]}': 1.0, # Positive: higher = better
    f'LEB{calibration_years[1]}': 0.2,            # Low weight for HDI
    f'EYE{calibration_years[1]}': 0.2,            # Low weight for HDI
    f'AEP{calibration_years[1]}': 0.2,            # Low weight for HDI
    f'gdp_{calibration_years[1]}': 1.3,           # Positive: higher = better
    f'urban_surface_{calibration_years[1]}': 1.0, # Positive: higher = better
    'health_dist': -1.6,                          # Negative: shorter = better
    'school_dist': -1.6                           # Negative: shorter = better
}

output_2015, output_2025 = run_hdi_downscaling(
    case_city=case_city,
    interim_path=interim_path,
    external_path=external_path,
    calibration_years=calibration_years,
    ref_raster_path=ref_raster_path,
    AOI_gdf=AOI_gdf,
    coord_weight=0.9,  # Tune this for spatial coherence
    feature_weights=feature_weights,
    output_suffix="_v05", 
    export_files=True
    )

🔍 Loading data...
  🔍 Loading grid data...
  📥 Extracting raster values for 2015...
  📥 Extracting raster values for 2025...
🎯 Clustering urban cells...
  🎯 Clustering 2015...
  🎯 Clustering 2025...
📊 Assigning SEP classes...
  📊 Assigning SEP classes...
💾 Exporting results...
  📄 Printing cluster centers...

📊 2015 Cluster Centers (Sorted by HDI):
    LEB2015   EYE2015   AEP2015  health_dist  school_dist  gdp_2015  urban_surface_2015  HDI_Mean
1  0.122169  0.140117  0.251672     1.174252     0.756050  2.285988            0.893983  0.171319
3  0.032213  0.033024 -0.039644     0.530425     0.626666 -0.441730            0.067549  0.008531
0 -0.124429 -0.146942 -0.121750    -1.342734    -1.138871 -0.830681           -0.767711 -0.131041
2 -0.293467 -0.260273 -0.037510    -4.804306    -5.145328 -0.934623           -0.823204 -0.197083

Suggested SEP class mapping (highest HDI → lowest):
  Cluster 0 → SEP Class 4
  Cluster 1 → SEP Class 3
  Cluster 2 → SEP Class 2
  Cluster 3 → SEP Class 1

📊

In [ ]:
# alternative: full workflow implementation
hdi_component_weight = [0.1,0.5]
gdp_weight = [1.0,1.8]
urbanisation_weight = 1.0
distances_weight = [-1.0,-1.5]

number_variations = 10

feature_weights = {
    f'LEB{calibration_years[0]}': hdi_component_weight,            # Low weight for HDI
    f'EYE{calibration_years[0]}': hdi_component_weight,            # Low weight for HDI
    f'AEP{calibration_years[0]}': hdi_component_weight,            # Low weight for HDI 
    f'gdp_{calibration_years[0]}': gdp_weight,           # Positive: higher = better
    f'urban_surface_{calibration_years[0]}': urbanisation_weight, # Positive: higher = better
    f'LEB{calibration_years[1]}': hdi_component_weight,            # Low weight for HDI
    f'EYE{calibration_years[1]}': hdi_component_weight,            # Low weight for HDI
    f'AEP{calibration_years[1]}': hdi_component_weight,            # Low weight for HDI
    f'gdp_{calibration_years[1]}': gdp_weight,           # Positive: higher = better
    f'urban_surface_{calibration_years[1]}': urbanisation_weight, # Positive: higher = better
    'health_dist': distances_weight,                          # Negative: shorter = better
    'school_dist': distances_weight                           # Negative: shorter = better
}
for i in range(number_variations):
    output_2015, output_2025 = run_hdi_downscaling(
        case_city=case_city,
        interim_path=interim_path,
        external_path=external_path,
        calibration_years=calibration_years,
        ref_raster_path=ref_raster_path,
        AOI_gdf=AOI_gdf,
        coord_weight=0.2,  # Tune this for spatial coherence, more means increased cluster coherence
        feature_weights=feature_weights,
        output_suffix=f"_v0{10+i}", 
        export_files=True
        )

🔍 Loading data...
  🔍 Loading grid data...
  📥 Extracting raster values for 2015...
  📥 Extracting raster values for 2025...
🎯 Clustering urban cells...
  🎯 Clustering 2015...
  🎯 Clustering 2025...
📊 Assigning SEP classes...
  📊 Assigning SEP classes...
💾 Exporting results...
  📄 Printing cluster centers...

📊 2015 Cluster Centers (Sorted by HDI):
    LEB2015   EYE2015   AEP2015  health_dist  school_dist  gdp_2015  urban_surface_2015  HDI_Mean
2  0.061429  0.069346  0.129789     1.125640     0.717842  3.809940            0.872028  0.086854
1  0.025920  0.030558 -0.003958     0.620692     0.695834 -0.459194            0.326061  0.017507
0 -0.049518 -0.061411 -0.059687    -0.659531    -0.597448 -1.233306           -0.784146 -0.056872
3 -0.102745 -0.099524 -0.043907    -3.926565    -3.625583 -1.428547           -0.826324 -0.082059

Suggested SEP class mapping (highest HDI → lowest):
  Cluster 0 → SEP Class 4
  Cluster 1 → SEP Class 3
  Cluster 2 → SEP Class 2
  Cluster 3 → SEP Class 1

📊

## implementing a loop to test through multiple value ranges

In [14]:
import itertools

# Define parameter ranges
hdi_weights = [0.1, 0.5]       # Low, High
gdp_weights = [1.0, 1.8]       # Low, High
distance_weights = [-1.0, -1.5] # Low, High magnitude

# Fixed weights
urbanisation_weight = 1.0
coord_weight = 0.2

# Iterate through all combinations
for i, (hdi_w, gdp_w, dist_w) in enumerate(itertools.product(hdi_weights, gdp_weights, distance_weights)):
    feature_weights = {
        # HDI components (same for both years)
        f'LEB{calibration_years[0]}': hdi_w,
        f'EYE{calibration_years[0]}': hdi_w,
        f'AEP{calibration_years[0]}': hdi_w,
        f'LEB{calibration_years[1]}': hdi_w,
        f'EYE{calibration_years[1]}': hdi_w,
        f'AEP{calibration_years[1]}': hdi_w,

        # GDP (same for both years)
        f'gdp_{calibration_years[0]}': gdp_w,
        f'gdp_{calibration_years[1]}': gdp_w,

        # Urbanisation (fixed)
        f'urban_surface_{calibration_years[0]}': urbanisation_weight,
        f'urban_surface_{calibration_years[1]}': urbanisation_weight,

        # Distances (same for both)
        'health_dist': dist_w,
        'school_dist': dist_w
    }

    # Generate descriptive suffix
    suffix = (
        f"_hdi{hdi_w}_gdp{gdp_w}_dist{abs(dist_w)}"  # e.g., "_hdi0.1_gdp1.8_dist1.5"
        f"_v{i+1}"  # Unique version number
    )

    output_2015, output_2025 = run_hdi_downscaling(
        case_city=case_city,
        interim_path=interim_path,
        external_path=external_path,
        calibration_years=calibration_years,
        ref_raster_path=ref_raster_path,
        AOI_gdf=AOI_gdf,
        coord_weight=coord_weight,
        feature_weights=feature_weights,
        output_suffix=suffix,
        export_files=True
    )

🔍 Loading data...
  🔍 Loading grid data...
  📥 Extracting raster values for 2015...
  📥 Extracting raster values for 2025...
🎯 Clustering urban cells...
  🎯 Clustering 2015...
  🎯 Clustering 2025...
📊 Assigning SEP classes...
  📊 Assigning SEP classes...
💾 Exporting results...
  📄 Printing cluster centers...

📊 2015 Cluster Centers (Sorted by HDI):
    LEB2015   EYE2015   AEP2015  health_dist  school_dist  gdp_2015  urban_surface_2015  HDI_Mean
1  0.061693  0.069645  0.130606     0.757026     0.488117  1.929355            0.926447  0.087314
0  0.023374  0.029087 -0.005175     0.355694     0.378565 -0.212642            0.665202  0.015762
2 -0.027342 -0.037976 -0.045978    -0.146001    -0.068761 -0.530795           -0.921090 -0.037099
3 -0.101479 -0.097655 -0.044655    -2.253400    -2.142190 -0.696455           -0.786466 -0.081263

Suggested SEP class mapping (highest HDI → lowest):
  Cluster 0 → SEP Class 4
  Cluster 1 → SEP Class 3
  Cluster 2 → SEP Class 2
  Cluster 3 → SEP Class 1

📊

# 5. Import the model calibration results and check calibration accuracy

Parameters to use in calibration:
- Urbanisation rate
- Proportion of SEP
  - Deprived
  - Low
  - Middle 
  - High
Workflow:
1. Load **agent** parameters: utility weight 
2. Load **model** parameters: agent population, slope, and other parameters
3. Save parameters in simulation run file using the 'run_report_template' sheet in "D:\GIT\ti-city-model\docs\TI_model_runs.xlsx"

In [27]:
# import the current model weights value
weights_file_name = 'weights.csv'
weights_path = ti_city_ascii_path / weights_file_name
#print(weights_path)
if weights_path.exists():
    weights_df = pd.read_csv(weights_path)
    print(f'Weights file found. Columns: {weights_df.columns.tolist()}')
weights_df.head()

Weights file found. Columns: ['agent', 'suburban', 'neighbour', 'bu', 'cbd', 'mall', 'markets', 'road', 'density', 'schools', 'attractive', 'health', 'random', 'water']


,agent,suburban,neighbour,bu,cbd,mall,markets,road,density,schools,attractive,health,random,water
0,dp,6,9,4,4,1,7,3,9,4,2,2,1,10
1,li,8,7,4,6,2,7,8,4,4,6,4,1,5
2,mi,6,7,7,7,7,7,6,4,4,7,4,1,5
3,hi,2,7,9,9,6,3,5,8,4,10,3,1,8
4,mi_red,7,0,0,0,0,0,8,0,5,7,0,0,0


In [ ]:
# import the latest results file


# The end (for now).